In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# Loading the data
df = pd.read_csv('PremierLeague.csv')


# Convert the 'Date' column to datetime format
# This is because we want to sort matches correctly in time

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.sort_values('Date').reset_index(drop=True)

/var/folders/db/bz186_t55wz58ywwqghvfvgr0000gp/T/ipykernel_2406/4100111796.py:12: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')


In [ ]:
# Converting match results into numeric values(1,0,-1)
# Oposite for the away team

df['HomeResult'] = df['FullTimeResult'].map({'H': 1, 'D': 0, 'A': -1})
df['AwayResult'] = df['FullTimeResult'].map({'H': -1, 'D': 0, 'A': 1})

# Calculate goal difference for each team
df['HomeGD'] = df['FullTimeHomeTeamGoals'] - df['FullTimeAwayTeamGoals']
df['AwayGD'] = df['FullTimeAwayTeamGoals'] - df['FullTimeHomeTeamGoals']

In [ ]:
# Create dataset for only home teams
home_df = df[['Date', 'HomeTeam', 'HomeResult', 'HomeGD']].copy()
home_df.columns = ['Date', 'Team', 'Result', 'GD']

# Create dataset for only away teams
away_df = df[['Date', 'AwayTeam', 'AwayResult', 'AwayGD']].copy()
away_df.columns = ['Date', 'Team', 'Result', 'GD']

# Combine both into one dataset where each row = one team in one match
teams = pd.concat([home_df, away_df])

# Sort by team and date so rolling calculations work correctly
teams = teams.sort_values(['Team', 'Date'])

In [20]:
weights = [0.2, 0.4, 0.6, 0.8, 1.0]

def weighted_score(x):
    if len(x) < 5:
        return None
    w = weights[-len(x):]
    return (x * w).sum()

teams['ResultScore'] = teams.groupby('Team')['Result'].rolling(5).apply(weighted_score).reset_index(level=0, drop=True)
teams['GDScore'] = teams.groupby('Team')['GD'].rolling(5).apply(weighted_score).reset_index(level=0, drop=True)

In [21]:
teams['Momentum'] = 0.6 * teams['ResultScore'] + 0.4 * teams['GDScore']

In [22]:
# Home momentum
home_mom = teams[['Date', 'Team', 'Momentum']].rename(columns={
    'Team': 'HomeTeam',
    'Momentum': 'HomeMomentum'
})

# Away momentum
away_mom = teams[['Date', 'Team', 'Momentum']].rename(columns={
    'Team': 'AwayTeam',
    'Momentum': 'AwayMomentum'
})

df = df.merge(home_mom, on=['Date', 'HomeTeam'], how='left')
df = df.merge(away_mom, on=['Date', 'AwayTeam'], how='left')

df['MomentumDiff'] = df['HomeMomentum'] - df['AwayMomentum']